# Build Gadr from raw-aligned inputs

This goal is to build the ADR–gene matrix from scratch in a careful, reproducible way.

It assumes the current project choice is:

- the label universe comes from `sider_lincs_labels_cid.csv`
- the gene universe comes from `gene_index_lincs.csv` produced by the finalized Gdrug notebook
- CTD and HPO are used as biological grounding sources for ADR ↔ gene edges

Outputs:

- `adr_index_full.csv`
- `adr_index.csv` (filtered ADR universe with at least one gene)
- `Gadr_full.npz`
- `Gadr_ctd.npz`
- `Gadr_hpo.npz`
- `Gadr.npz`
- `Gadr_summary.json`

If your file names differ, update the path cell below.

In [6]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz
import json
import re

## HPO --> ADR name space matching to SIDER

In [17]:
labels = pd.read_csv("../data_2/sider_lincs_labels_cid.csv")
hpo= pd.read_csv("F:/CDSS/prototype4/raw_data/adr_hpo_gene_grounding.csv")

print("labels:", labels.shape)
print("hpo_raw:", hpo.shape)
print("labels columns:", labels.columns.tolist())
print("hpo_raw columns:", hpo.columns.tolist())

labels: (106173, 8)
hpo_raw: (209758, 7)
labels columns: ['drug_id', 'cid', 'drug_name', 'drug_name_norm_x', 'pert_id', 'pert_iname', 'adr_id', 'adr_name']
hpo_raw columns: ['Unnamed: 0', 'adr_id', 'adr_name', 'HPO_ID', 'hpo_name', 'gene_symbol', 'ncbi_gene_id']


In [18]:
DRUG_SALT_WORDS = [
    "hydrochloride", "sodium", "potassium", "acetate",
    "sulfate", "phosphate", "tartrate", "succinate",
    "fumarate", "nitrate"
]


def normalize_drug_text(x):
    if pd.isna(x):
        return None

    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9\s]", " ", x)

    for w in DRUG_SALT_WORDS:
        x = re.sub(rf"\b{re.escape(w)}\b", " ", x)

    x = re.sub(r"\s+", " ", x).strip()
    return x if x else None

def normalize_adr_text(x):
    if pd.isna(x):
        return None

    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x if x else None

def normalize_gene(x):
    if pd.isna(x):
        return None

    x = str(x).upper().strip()
    x = re.sub(r"\s+", " ", x)
    return x if x else None

def clean_id(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None

In [20]:
labels = labels.copy()

labels["adr_id"] = labels["adr_id"].map(clean_id)
labels["adr_name"] = labels["adr_name"].astype(str).str.strip()
labels["adr_name_norm"] = labels["adr_name"].map(normalize_adr_text)
hpo["adr_name_norm"] = hpo["adr_name"].map(normalize_adr_text)

adr_vocab = (
    labels[["adr_id", "adr_name", "adr_name_norm"]]
    .dropna(subset=["adr_id", "adr_name", "adr_name_norm"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print("ADR vocab shape:", adr_vocab.shape)
print("Unique ADR IDs:", adr_vocab["adr_id"].nunique())
print("Unique normalized ADR names:", adr_vocab["adr_name_norm"].nunique())

adr_vocab.head()

ADR vocab shape: (3366, 3)
Unique ADR IDs: 3366
Unique normalized ADR names: 3366


,adr_id,adr_name,adr_name_norm
0,C0002170,Alopecia,alopecia
1,C0232462,Decreased appetite,decreased appetite
2,C0009450,Infection,infection
3,C0009806,Constipation,constipation
4,C0011603,Dermatitis,dermatitis


In [21]:
# If multiple ADR IDs collapse to the same normalized name,
# keep the first one for automatic mapping.
# We will inspect collisions separately.

adr_name_map = (
    adr_vocab
    .drop_duplicates(subset=["adr_name_norm"])
    .set_index("adr_name_norm")[["adr_id", "adr_name"]]
)

print("Normalized ADR map size:", adr_name_map.shape)

Normalized ADR map size: (3366, 2)


In [22]:
# Inspect collisions in the SIDER ADR namespace after normalization

adr_collisions = (
    adr_vocab.groupby("adr_name_norm")
    .agg(
        n_adr_ids=("adr_id", "nunique"),
        adr_ids=("adr_id", lambda x: sorted(set(x))[:10]),
        adr_names=("adr_name", lambda x: sorted(set(x))[:10])
    )
    .reset_index()
)

adr_collisions = adr_collisions[adr_collisions["n_adr_ids"] > 1].sort_values("n_adr_ids", ascending=False)

print("Number of normalized-name collisions in ADR vocab:", len(adr_collisions))
adr_collisions.head(20)

Number of normalized-name collisions in ADR vocab: 0


,adr_name_norm,n_adr_ids,adr_ids,adr_names


In [23]:
if "Unnamed: 0" in hpo.columns:
    hpo = hpo.drop(columns=["Unnamed: 0"])

required_hpo_cols = {"adr_name", "gene_symbol"}
assert required_hpo_cols.issubset(hpo.columns), required_hpo_cols - set(hpo.columns)

hpo["adr_name_raw"] = hpo["adr_name"].astype(str).str.strip()
hpo["gene_symbol"] = hpo["gene_symbol"].map(normalize_gene)
hpo["adr_name_norm"] = hpo["adr_name_raw"].map(normalize_adr_text)

hpo = hpo.dropna(subset=["adr_name_raw", "gene_symbol", "adr_name_norm"]).reset_index(drop=True)

print("HPO after basic cleaning:", hpo.shape)
hpo.head()

HPO after basic cleaning: (209758, 8)


,adr_id,adr_name,HPO_ID,hpo_name,gene_symbol,ncbi_gene_id,adr_name_norm,adr_name_raw
0,C0002170,Alopecia,HP:0007534,Congenital posterior occipital alopecia,HRAS,3265,alopecia,Alopecia
1,C0232462,Decreased appetite,HP:0004396,Poor appetite,KRT10,3858,decreased appetite,Decreased appetite
2,C0232462,Decreased appetite,HP:0004396,Poor appetite,KRT1,3848,decreased appetite,Decreased appetite
3,C0232462,Decreased appetite,HP:0004396,Poor appetite,IL18BP,10068,decreased appetite,Decreased appetite
4,C0232462,Decreased appetite,HP:0004396,Poor appetite,TRANK1,9881,decreased appetite,Decreased appetite


In [25]:
adr_name_map = (
    adr_vocab
    .drop_duplicates(subset=["adr_name_norm"])
    .set_index("adr_name_norm")[["adr_id", "adr_name"]]
)

print("Normalized ADR map size:", adr_name_map.shape)
adr_name_map.head()

Normalized ADR map size: (3366, 2)


,adr_id,adr_name
adr_name_norm,,
alopecia,C0002170,Alopecia
decreased appetite,C0232462,Decreased appetite
infection,C0009450,Infection
constipation,C0009806,Constipation
dermatitis,C0011603,Dermatitis


In [26]:
hpo["mapped_adr_id"] = hpo["adr_name_norm"].map(adr_name_map["adr_id"])
hpo["mapped_adr_name"] = hpo["adr_name_norm"].map(adr_name_map["adr_name"])

print("Mapped rows:", hpo["mapped_adr_id"].notna().sum())
print("Unmapped rows:", hpo["mapped_adr_id"].isna().sum())
print("Mapped unique ADRs:", hpo["mapped_adr_id"].dropna().nunique())
print("Mapped unique genes:", hpo.loc[hpo["mapped_adr_id"].notna(), "gene_symbol"].nunique())

Mapped rows: 190393
Unmapped rows: 19365
Mapped unique ADRs: 1963
Mapped unique genes: 4995


In [27]:
hpo_mapped_debug = hpo[
    [
        "adr_name_raw",
        "adr_name_norm",
        "mapped_adr_id",
        "mapped_adr_name",
        "gene_symbol"
    ]
].drop_duplicates().reset_index(drop=True)

print("Mapped debug table shape:", hpo_mapped_debug.shape)
hpo_mapped_debug.head()

Mapped debug table shape: (209758, 5)


,adr_name_raw,adr_name_norm,mapped_adr_id,mapped_adr_name,gene_symbol
0,Alopecia,alopecia,C0002170,Alopecia,HRAS
1,Decreased appetite,decreased appetite,C0232462,Decreased appetite,KRT10
2,Decreased appetite,decreased appetite,C0232462,Decreased appetite,KRT1
3,Decreased appetite,decreased appetite,C0232462,Decreased appetite,IL18BP
4,Decreased appetite,decreased appetite,C0232462,Decreased appetite,TRANK1


In [28]:
adr_hpo_gene_grounding_clean = (
    hpo_mapped_debug
    .dropna(subset=["mapped_adr_id", "mapped_adr_name"])
    .rename(columns={
        "mapped_adr_id": "adr_id",
        "mapped_adr_name": "adr_name"
    })
    [["adr_id", "adr_name", "gene_symbol"]]
    .drop_duplicates()
    .sort_values(["adr_id", "gene_symbol"])
    .reset_index(drop=True)
)

print("Final HPO clean shape:", adr_hpo_gene_grounding_clean.shape)
print("Unique ADRs:", adr_hpo_gene_grounding_clean["adr_id"].nunique())
print("Unique genes:", adr_hpo_gene_grounding_clean["gene_symbol"].nunique())

adr_hpo_gene_grounding_clean.head()

Final HPO clean shape: (190393, 3)
Unique ADRs: 1963
Unique genes: 4995


,adr_id,adr_name,gene_symbol
0,C0000727,Acute abdomen,ACY1
1,C0000727,Acute abdomen,ATP5F1A
2,C0000727,Acute abdomen,COG8
3,C0000727,Acute abdomen,FBP1
4,C0000727,Acute abdomen,GCDH


In [29]:
hpo_unmatched = (
    hpo_mapped_debug[hpo_mapped_debug["mapped_adr_id"].isna()]
    [["adr_name_raw", "adr_name_norm"]]
    .drop_duplicates()
    .sort_values(["adr_name_norm", "adr_name_raw"])
    .reset_index(drop=True)
)

print("Unmatched unique HPO terms:", len(hpo_unmatched))
hpo_unmatched.head(30)

Unmatched unique HPO terms: 205


,adr_name_raw,adr_name_norm
0,Abdominal injury,abdominal injury
1,Abdominal neoplasm,abdominal neoplasm
2,Acne cystic,acne cystic
3,Acne pustular,acne pustular
4,Alanine aminotransferase decreased,alanine aminotransferase decreased
5,Alcohol problem,alcohol problem
6,Allergic sinusitis,allergic sinusitis
7,Amaurosis fugax,amaurosis fugax
8,Amniotic cavity infection,amniotic cavity infection
9,Anaplastic large-cell lymphoma,anaplastic large cell lymphoma


In [30]:
hpo_mapped_debug.to_csv("../data_2/adr_hpo_gene_grounding_mapped_debug.csv", index=False)
adr_hpo_gene_grounding_clean.to_csv("../data_2/adr_hpo_gene_grounding_clean.csv", index=False)

print("Saved debug and final HPO files.")

Saved debug and final HPO files.


In [31]:
hpo_mapped_debug.head()

,adr_name_raw,adr_name_norm,mapped_adr_id,mapped_adr_name,gene_symbol
0,Alopecia,alopecia,C0002170,Alopecia,HRAS
1,Decreased appetite,decreased appetite,C0232462,Decreased appetite,KRT10
2,Decreased appetite,decreased appetite,C0232462,Decreased appetite,KRT1
3,Decreased appetite,decreased appetite,C0232462,Decreased appetite,IL18BP
4,Decreased appetite,decreased appetite,C0232462,Decreased appetite,TRANK1


In [32]:
adr_hpo_gene_grounding_clean.head()

,adr_id,adr_name,gene_symbol
0,C0000727,Acute abdomen,ACY1
1,C0000727,Acute abdomen,ATP5F1A
2,C0000727,Acute abdomen,COG8
3,C0000727,Acute abdomen,FBP1
4,C0000727,Acute abdomen,GCDH


## CTD-->ADR name space matching to SIDER

In [33]:
ctd_raw = pd.read_csv("F:/CDSS/prototype4/raw_data/CTD_curated_genes_diseases_human_only.tsv", sep="\t", comment="#")

print("labels:", labels.shape)
print("ctd_raw:", ctd_raw.shape)
print("labels columns:", labels.columns.tolist())
print("ctd_raw columns:", ctd_raw.columns.tolist())

labels: (106173, 9)
ctd_raw: (35429, 7)
labels columns: ['drug_id', 'cid', 'drug_name', 'drug_name_norm_x', 'pert_id', 'pert_iname', 'adr_id', 'adr_name', 'adr_name_norm']
ctd_raw columns: ['GeneSymbol', 'GeneID', 'DiseaseName', 'DiseaseID', 'DirectEvidence', 'InferenceGeneSymbol', 'PubMedIDs']


In [34]:
def normalize_adr_text(x):
    if pd.isna(x):
        return None

    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x if x else None

def normalize_gene(x):
    if pd.isna(x):
        return None

    x = str(x).upper().strip()
    x = re.sub(r"\s+", " ", x)
    return x if x else None

def clean_id(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None

In [35]:
labels = labels.copy()

labels["adr_id"] = labels["adr_id"].map(clean_id)
labels["adr_name"] = labels["adr_name"].astype(str).str.strip()
labels["adr_name_norm"] = labels["adr_name"].map(normalize_adr_text)

adr_vocab = (
    labels[["adr_id", "adr_name", "adr_name_norm"]]
    .dropna(subset=["adr_id", "adr_name", "adr_name_norm"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print("ADR vocab shape:", adr_vocab.shape)
print("Unique ADR IDs:", adr_vocab["adr_id"].nunique())
print("Unique normalized ADR names:", adr_vocab["adr_name_norm"].nunique())

adr_vocab.head()

ADR vocab shape: (3366, 3)
Unique ADR IDs: 3366
Unique normalized ADR names: 3366


,adr_id,adr_name,adr_name_norm
0,C0002170,Alopecia,alopecia
1,C0232462,Decreased appetite,decreased appetite
2,C0009450,Infection,infection
3,C0009806,Constipation,constipation
4,C0011603,Dermatitis,dermatitis


In [37]:
adr_name_map = (
    adr_vocab
    .drop_duplicates(subset=["adr_name_norm"])
    .set_index("adr_name_norm")[["adr_id", "adr_name"]]
)

print("Normalized ADR map size:", adr_name_map.shape)

Normalized ADR map size: (3366, 2)


In [38]:
required_ctd_cols = {"GeneSymbol", "DiseaseName"}
assert required_ctd_cols.issubset(ctd_raw.columns), required_ctd_cols - set(ctd_raw.columns)

In [39]:
ctd = ctd_raw.copy()

ctd["GeneSymbol"] = ctd["GeneSymbol"].map(normalize_gene)
ctd["DiseaseName_raw"] = ctd["DiseaseName"].astype(str).str.strip()
ctd["DiseaseName_norm"] = ctd["DiseaseName_raw"].map(normalize_adr_text)

if "DiseaseID" in ctd.columns:
    ctd["DiseaseID"] = ctd["DiseaseID"].map(clean_id)

if "DirectEvidence" in ctd.columns:
    ctd["DirectEvidence"] = ctd["DirectEvidence"].astype(str).str.strip()

ctd = ctd.dropna(subset=["GeneSymbol", "DiseaseName_raw", "DiseaseName_norm"]).reset_index(drop=True)

print("CTD after basic cleaning:", ctd.shape)
ctd.head()

CTD after basic cleaning: (35429, 9)


,GeneSymbol,GeneID,DiseaseName,DiseaseID,DirectEvidence,InferenceGeneSymbol,PubMedIDs,DiseaseName_raw,DiseaseName_norm
0,A,50518,Dermatitis,MESH:D003872,marker/mechanism,NaN,32937126,Dermatitis,dermatitis
1,A,50518,Diabetes Mellitus,MESH:D003920,marker/mechanism,NaN,1473152|25620042,Diabetes Mellitus,diabetes mellitus
2,A,50518,"Diabetes Mellitus, Type 2",MESH:D003924,marker/mechanism,NaN,8146154,"Diabetes Mellitus, Type 2",diabetes mellitus type 2
3,A,50518,Diabetic Nephropathies,MESH:D003928,marker/mechanism,NaN,37769864,Diabetic Nephropathies,diabetic nephropathies
4,A,50518,Edema,MESH:D004487,marker/mechanism,NaN,32937126,Edema,edema


In [40]:
# Optional inspection of DirectEvidence values
if "DirectEvidence" in ctd.columns:
    print(ctd["DirectEvidence"].fillna("NA").value_counts(dropna=False).head(20))

DirectEvidence
marker/mechanism                33190
therapeutic                      1931
marker/mechanism|therapeutic      308
Name: count, dtype: int64


In [41]:
# Keep all rows for now
# If later we want direct-evidence-only, we can filter and compare

ctd["mapped_adr_id"] = ctd["DiseaseName_norm"].map(adr_name_map["adr_id"])
ctd["mapped_adr_name"] = ctd["DiseaseName_norm"].map(adr_name_map["adr_name"])

print("Mapped rows:", ctd["mapped_adr_id"].notna().sum())
print("Unmapped rows:", ctd["mapped_adr_id"].isna().sum())
print("Mapped unique ADRs:", ctd["mapped_adr_id"].dropna().nunique())
print("Mapped unique genes:", ctd.loc[ctd["mapped_adr_id"].notna(), "GeneSymbol"].nunique())

Mapped rows: 5974
Unmapped rows: 29455
Mapped unique ADRs: 371
Mapped unique genes: 2963


In [42]:
ctd_mapped_debug_cols = [
    "DiseaseName_raw",
    "DiseaseName_norm",
    "mapped_adr_id",
    "mapped_adr_name",
    "GeneSymbol"
]

for col in ["DiseaseID", "DirectEvidence", "PubMedIDs"]:
    if col in ctd.columns:
        ctd_mapped_debug_cols.append(col)

ctd_mapped_debug = (
    ctd[ctd_mapped_debug_cols]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("CTD mapped debug shape:", ctd_mapped_debug.shape)
ctd_mapped_debug.head()

CTD mapped debug shape: (35429, 8)


,DiseaseName_raw,DiseaseName_norm,mapped_adr_id,mapped_adr_name,GeneSymbol,DiseaseID,DirectEvidence,PubMedIDs
0,Dermatitis,dermatitis,C0011603,Dermatitis,A,MESH:D003872,marker/mechanism,32937126
1,Diabetes Mellitus,diabetes mellitus,C0011849,Diabetes mellitus,A,MESH:D003920,marker/mechanism,1473152|25620042
2,"Diabetes Mellitus, Type 2",diabetes mellitus type 2,NaN,NaN,A,MESH:D003924,marker/mechanism,8146154
3,Diabetic Nephropathies,diabetic nephropathies,NaN,NaN,A,MESH:D003928,marker/mechanism,37769864
4,Edema,edema,NaN,NaN,A,MESH:D004487,marker/mechanism,32937126


In [43]:
adr_ctd_disease_genes = (
    ctd_mapped_debug
    .dropna(subset=["mapped_adr_id", "mapped_adr_name"])
    .rename(columns={
        "mapped_adr_id": "adr_id",
        "mapped_adr_name": "adr_name",
        "GeneSymbol": "GeneSymbol"
    })
    [["adr_id", "adr_name", "GeneSymbol"]]
    .drop_duplicates()
    .sort_values(["adr_id", "GeneSymbol"])
    .reset_index(drop=True)
)

print("Final CTD clean shape:", adr_ctd_disease_genes.shape)
print("Unique ADRs:", adr_ctd_disease_genes["adr_id"].nunique())
print("Unique genes:", adr_ctd_disease_genes["GeneSymbol"].nunique())

adr_ctd_disease_genes.head()

Final CTD clean shape: (5974, 3)
Unique ADRs: 371
Unique genes: 2963


,adr_id,adr_name,GeneSymbol
0,C0000737,Abdominal pain,IFNA2
1,C0000786,Abortion spontaneous,ACE2
2,C0000786,Abortion spontaneous,ACKR4
3,C0000786,Abortion spontaneous,AGTR1
4,C0000786,Abortion spontaneous,AHR


In [45]:
ctd_unmatched = (
    ctd_mapped_debug[ctd_mapped_debug["mapped_adr_id"].isna()]
    [["DiseaseName_raw", "DiseaseName_norm"]]
    .drop_duplicates()
    .sort_values(["DiseaseName_norm", "DiseaseName_raw"])
    .reset_index(drop=True)
)

print("Unmatched unique CTD disease terms:", len(ctd_unmatched))
ctd_unmatched.head(30)

Unmatched unique CTD disease terms: 5484


,DiseaseName_raw,DiseaseName_norm
0,17-Hydroxysteroid Dehydrogenase Deficiency,17 hydroxysteroid dehydrogenase deficiency
1,18-Hydroxylase deficiency,18 hydroxylase deficiency
2,"2,4-Dienoyl-CoA Reductase Deficiency",2 4 dienoyl coa reductase deficiency
3,2-Hydroxyglutaricaciduria,2 hydroxyglutaricaciduria
4,2-Methylbutyryl-CoA Dehydrogenase Deficiency,2 methylbutyryl coa dehydrogenase deficiency
5,22q11 Deletion Syndrome,22q11 deletion syndrome
6,3-Hydroxy-3-Methylglutaryl-CoA Lyase Deficiency,3 hydroxy 3 methylglutaryl coa lyase deficiency
7,3-Hydroxy-3-Methylglutaryl-CoA Synthase 2 Defi...,3 hydroxy 3 methylglutaryl coa synthase 2 defi...
8,3-Hydroxyacyl-CoA Dehydrogenase Deficiency,3 hydroxyacyl coa dehydrogenase deficiency
9,3-methylcrotonyl CoA carboxylase 1 deficiency,3 methylcrotonyl coa carboxylase 1 deficiency


In [46]:
ctd_mapped_debug.to_csv("../data_2/adr_ctd_disease_genes_mapped_debug.csv", index=False)
adr_ctd_disease_genes.to_csv("../data_2/adr_ctd_disease_genes.csv", index=False)
ctd_unmatched.to_csv("../data_2/adr_ctd_unmatched_terms.csv", index=False)



In [47]:
summary = {
    "ctd_raw_rows": int(len(ctd_raw)),
    "ctd_clean_rows": int(len(ctd)),
    "ctd_debug_rows": int(len(ctd_mapped_debug)),
    "ctd_final_rows": int(len(adr_ctd_disease_genes)),
    "ctd_final_unique_adrs": int(adr_ctd_disease_genes["adr_id"].nunique()) if len(adr_ctd_disease_genes) > 0 else 0,
    "ctd_final_unique_genes": int(adr_ctd_disease_genes["GeneSymbol"].nunique()) if len(adr_ctd_disease_genes) > 0 else 0,
    "ctd_unmatched_unique_terms": int(len(ctd_unmatched))
}

summary

{'ctd_raw_rows': 35429,
 'ctd_clean_rows': 35429,
 'ctd_debug_rows': 35429,
 'ctd_final_rows': 5974,
 'ctd_final_unique_adrs': 371,
 'ctd_final_unique_genes': 2963,
 'ctd_unmatched_unique_terms': 5484}

## matrix construction

In [48]:
import json
from scipy.sparse import csr_matrix, save_npz

In [60]:
LABELS_PATH = "../data_2/sider_lincs_labels_cid.csv"
HPO_CLEAN_PATH = "../data_2/adr_hpo_gene_grounding_clean.csv"
CTD_CLEAN_PATH = "../data_2/adr_ctd_disease_genes.csv"
GENE_INDEX_PATH = "../data_2/gene_index_lincs.csv"

In [61]:
labels = pd.read_csv(LABELS_PATH)
hpo = pd.read_csv(HPO_CLEAN_PATH)
ctd = pd.read_csv(CTD_CLEAN_PATH)
gene_lincs = pd.read_csv(GENE_INDEX_PATH)

print("labels:", labels.shape)
print("hpo:", hpo.shape)
print("ctd:", ctd.shape)
print("gene_lincs:", gene_lincs.shape)

labels: (106173, 8)
hpo: (190393, 3)
ctd: (5974, 3)
gene_lincs: (978, 3)


In [62]:
required_label_cols = {"pert_id", "adr_id", "adr_name"}
required_hpo_cols = {"adr_id", "adr_name", "gene_symbol"}
required_ctd_cols = {"adr_id", "adr_name", "GeneSymbol"}
required_gene_cols = {"gene_idx", "GeneSymbol"}

assert required_label_cols.issubset(labels.columns), required_label_cols - set(labels.columns)
assert required_hpo_cols.issubset(hpo.columns), required_hpo_cols - set(hpo.columns)
assert required_ctd_cols.issubset(ctd.columns), required_ctd_cols - set(ctd.columns)
assert required_gene_cols.issubset(gene_lincs.columns), required_gene_cols - set(gene_lincs.columns)

print("All required columns are present.")

All required columns are present.


In [63]:
def clean_id(s):
    return s.astype(str).str.strip()

def clean_text(s):
    return (
        s.astype(str)
         .str.strip()
         .str.replace(r"\s+", " ", regex=True)
    )

def clean_gene(s):
    return (
        s.astype(str)
         .str.upper()
         .str.strip()
         .str.replace(r"\s+", " ", regex=True)
    )

In [64]:
labels = labels.copy()
labels["pert_id"] = clean_id(labels["pert_id"])
labels["adr_id"] = clean_id(labels["adr_id"])
labels["adr_name"] = clean_text(labels["adr_name"])
labels = labels.dropna(subset=["pert_id", "adr_id", "adr_name"]).drop_duplicates().reset_index(drop=True)

hpo = hpo.copy()
hpo["adr_id"] = clean_id(hpo["adr_id"])
hpo["adr_name"] = clean_text(hpo["adr_name"])
hpo["gene_symbol"] = clean_gene(hpo["gene_symbol"])
hpo = hpo.dropna(subset=["adr_id", "adr_name", "gene_symbol"]).drop_duplicates().reset_index(drop=True)

ctd = ctd.copy()
ctd["adr_id"] = clean_id(ctd["adr_id"])
ctd["adr_name"] = clean_text(ctd["adr_name"])
ctd["GeneSymbol"] = clean_gene(ctd["GeneSymbol"])
ctd = ctd.dropna(subset=["adr_id", "adr_name", "GeneSymbol"]).drop_duplicates().reset_index(drop=True)

gene_lincs = gene_lincs.copy()
gene_lincs["GeneSymbol"] = clean_gene(gene_lincs["GeneSymbol"])
gene_lincs = gene_lincs[["gene_idx", "GeneSymbol"]].drop_duplicates().sort_values("gene_idx").reset_index(drop=True)

assert (gene_lincs["gene_idx"].to_numpy() == np.arange(len(gene_lincs))).all()

print("labels cleaned:", labels.shape)
print("hpo cleaned:", hpo.shape)
print("ctd cleaned:", ctd.shape)
print("gene_lincs cleaned:", gene_lincs.shape)

labels cleaned: (106173, 8)
hpo cleaned: (190393, 3)
ctd cleaned: (5974, 3)
gene_lincs cleaned: (978, 2)


In [65]:
adr_index_full = (
    labels[["adr_id", "adr_name"]]
    .drop_duplicates()
    .sort_values(["adr_id", "adr_name"])
    .reset_index(drop=True)
)

adr_index_full["adr_idx"] = np.arange(len(adr_index_full))
adr_index_full = adr_index_full[["adr_idx", "adr_id", "adr_name"]]

print("Full ADR universe:", len(adr_index_full))
adr_index_full.head()

Full ADR universe: 3366


,adr_idx,adr_id,adr_name
0,0,C0000727,Acute abdomen
1,1,C0000731,Abdominal distension
2,2,C0000737,Abdominal pain
3,3,C0000768,Congenital anomaly
4,4,C0000786,Abortion spontaneous


In [66]:
valid_adrs = set(adr_index_full["adr_id"])
valid_genes = set(gene_lincs["GeneSymbol"])

print("HPO ADR overlap:", len(set(hpo["adr_id"]) & valid_adrs))
print("CTD ADR overlap:", len(set(ctd["adr_id"]) & valid_adrs))
print("HPO gene overlap:", len(set(hpo["gene_symbol"]) & valid_genes))
print("CTD gene overlap:", len(set(ctd["GeneSymbol"]) & valid_genes))

HPO ADR overlap: 1963
CTD ADR overlap: 371
HPO gene overlap: 364
CTD gene overlap: 265


In [70]:
hpo_m = (
    hpo
    .merge(adr_index_full[["adr_id", "adr_idx"]], on="adr_id", how="inner")
    .merge(gene_lincs, left_on="gene_symbol", right_on="GeneSymbol", how="inner")
    [["adr_idx", "gene_idx"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Indexed HPO edges:", hpo_m.shape)
hpo_m.head()

Indexed HPO edges: (16738, 2)


,adr_idx,gene_idx
0,0,478
1,0,488
2,0,636
3,1,128
4,1,198


In [71]:
ctd_m = (
    ctd
    .merge(adr_index_full[["adr_id", "adr_idx"]], on="adr_id", how="inner")
    .merge(gene_lincs, on="GeneSymbol", how="inner")
    [["adr_idx", "gene_idx"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Indexed CTD edges:", ctd_m.shape)
ctd_m.head()

Indexed CTD edges: (774, 2)


,adr_idx,gene_idx
0,4,384
1,4,129
2,4,387
3,4,392
4,4,750


In [72]:
rows = hpo_m["adr_idx"].to_numpy()
cols = hpo_m["gene_idx"].to_numpy()
data = np.ones(len(hpo_m), dtype=np.int8)

Gadr_hpo = csr_matrix(
    (data, (rows, cols)),
    shape=(len(adr_index_full), len(gene_lincs))
)

rows = ctd_m["adr_idx"].to_numpy()
cols = ctd_m["gene_idx"].to_numpy()
data = np.ones(len(ctd_m), dtype=np.int8)

Gadr_ctd = csr_matrix(
    (data, (rows, cols)),
    shape=(len(adr_index_full), len(gene_lincs))
)

Gadr_full = Gadr_hpo.maximum(Gadr_ctd)

print("Gadr_hpo shape:", Gadr_hpo.shape, "nnz:", Gadr_hpo.nnz)
print("Gadr_ctd shape:", Gadr_ctd.shape, "nnz:", Gadr_ctd.nnz)
print("Gadr_full shape:", Gadr_full.shape, "nnz:", Gadr_full.nnz)

Gadr_hpo shape: (3366, 978) nnz: 16738
Gadr_ctd shape: (3366, 978) nnz: 774
Gadr_full shape: (3366, 978) nnz: 17481


In [73]:
genes_per_adr_full = np.asarray(Gadr_full.sum(axis=1)).ravel()
adrs_per_gene_full = np.asarray(Gadr_full.sum(axis=0)).ravel()

print("ADRs:", Gadr_full.shape[0])
print("Genes:", Gadr_full.shape[1])
print("Union nnz:", Gadr_full.nnz)
print("Density:", Gadr_full.nnz / (Gadr_full.shape[0] * Gadr_full.shape[1]))
print("Zero-gene ADRs:", int((genes_per_adr_full == 0).sum()))
print("Mean genes / ADR:", float(genes_per_adr_full.mean()))
print("Median genes / ADR:", float(np.median(genes_per_adr_full)))
print("Max genes / ADR:", int(genes_per_adr_full.max()))
print("Mean ADRs / gene:", float(adrs_per_gene_full.mean()))
print("Median ADRs / gene:", float(np.median(adrs_per_gene_full)))
print("Max ADRs / gene:", int(adrs_per_gene_full.max()))

ADRs: 3366
Genes: 978
Union nnz: 17481
Density: 0.005310229687710741
Zero-gene ADRs: 2019
Mean genes / ADR: 5.193404634581105
Median genes / ADR: 0.0
Max genes / ADR: 319
Mean ADRs / gene: 17.874233128834355
Median ADRs / gene: 0.0
Max ADRs / gene: 314


In [74]:
adr_index_full.to_csv("../data_2/adr_index_full.csv", index=False)
save_npz("../data_2/Gadr_full.npz", Gadr_full)
save_npz("../data_2/Gadr_hpo.npz", Gadr_hpo)
save_npz("../data_2/Gadr_ctd.npz", Gadr_ctd)

print("Saved full ADR-gene artifacts.")

Saved full ADR-gene artifacts.


In [75]:
valid_mask = genes_per_adr_full > 0

Gadr = Gadr_full[valid_mask]

adr_index = adr_index_full.loc[valid_mask].reset_index(drop=True)
adr_index["adr_idx"] = np.arange(len(adr_index))
adr_index = adr_index[["adr_idx", "adr_id", "adr_name"]]

genes_per_adr = np.asarray(Gadr.sum(axis=1)).ravel()
adrs_per_gene = np.asarray(Gadr.sum(axis=0)).ravel()

print("Filtered ADRs:", Gadr.shape[0])
print("Filtered genes:", Gadr.shape[1])
print("Filtered nnz:", Gadr.nnz)
print("Filtered density:", Gadr.nnz / (Gadr.shape[0] * Gadr.shape[1]))
print("Mean genes / ADR:", float(genes_per_adr.mean()))
print("Median genes / ADR:", float(np.median(genes_per_adr)))
print("Max genes / ADR:", int(genes_per_adr.max()))
print("Min genes / ADR:", int(genes_per_adr.min()))
print("Mean ADRs / gene:", float(adrs_per_gene.mean()))
print("Median ADRs / gene:", float(np.median(adrs_per_gene)))
print("Max ADRs / gene:", int(adrs_per_gene.max()))

Filtered ADRs: 1347
Filtered genes: 978
Filtered nnz: 17481
Filtered density: 0.01326966082318809
Mean genes / ADR: 12.977728285077951
Median genes / ADR: 4.0
Max genes / ADR: 319
Min genes / ADR: 1
Mean ADRs / gene: 17.874233128834355
Median ADRs / gene: 0.0
Max ADRs / gene: 314


In [76]:
adr_index.to_csv("../data_2/adr_index_filtered.csv", index=False)
save_npz("../data_2/Gadr_filtered.npz", Gadr)

print("Saved filtered Gadr artifacts.")

Saved filtered Gadr artifacts.


In [77]:
valid_adr_ids = set(adr_index["adr_id"])

labels_filtered = (
    labels[labels["adr_id"].isin(valid_adr_ids)]
    .copy()
    .reset_index(drop=True)
)

print("Original label rows:", len(labels))
print("Filtered label rows:", len(labels_filtered))
print("Original ADR count:", labels["adr_id"].nunique())
print("Filtered ADR count:", labels_filtered["adr_id"].nunique())

labels_filtered.to_csv("../data_2/sider_lincs_labels_cid_filtered_by_gadr.csv", index=False)

print("Saved filtered labels.")

Original label rows: 106173
Filtered label rows: 57030
Original ADR count: 3366
Filtered ADR count: 1347
Saved filtered labels.


In [78]:
summary = {
    "full_n_adr": int(Gadr_full.shape[0]),
    "filtered_n_adr": int(Gadr.shape[0]),
    "n_gene": int(Gadr.shape[1]),
    "hpo_nnz": int(Gadr_hpo.nnz),
    "ctd_nnz": int(Gadr_ctd.nnz),
    "full_nnz": int(Gadr_full.nnz),
    "filtered_nnz": int(Gadr.nnz),
    "full_density": float(Gadr_full.nnz / (Gadr_full.shape[0] * Gadr_full.shape[1])),
    "filtered_density": float(Gadr.nnz / (Gadr.shape[0] * Gadr.shape[1])),
    "zero_gene_adrs_removed": int((genes_per_adr_full == 0).sum()),
    "mean_genes_per_adr_filtered": float(genes_per_adr.mean()),
    "median_genes_per_adr_filtered": float(np.median(genes_per_adr)),
    "max_genes_per_adr_filtered": int(genes_per_adr.max())
}

with open("../data_2/Gadr_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

summary

{'full_n_adr': 3366,
 'filtered_n_adr': 1347,
 'n_gene': 978,
 'hpo_nnz': 16738,
 'ctd_nnz': 774,
 'full_nnz': 17481,
 'filtered_nnz': 17481,
 'full_density': 0.005310229687710741,
 'filtered_density': 0.01326966082318809,
 'zero_gene_adrs_removed': 2019,
 'mean_genes_per_adr_filtered': 12.977728285077951,
 'median_genes_per_adr_filtered': 4.0,
 'max_genes_per_adr_filtered': 319}

In [79]:
assert Gadr_full.shape[1] == len(gene_lincs)
assert Gadr.shape[1] == len(gene_lincs)
assert len(adr_index_full) == Gadr_full.shape[0]
assert len(adr_index) == Gadr.shape[0]
assert Gadr_full.nnz > 0
assert Gadr.nnz > 0

print("All consistency checks passed.")

All consistency checks passed.
